# Inferencia para Unsloth (solo funciona para GPU)

In [1]:
# CÓDIGO DE ESTA CELDA SACADO DE LA DOCUMENTACIÓN DE UNSLOTH
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.7/290.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.12.7 req

### Configuración global (para el resto de procesos de inferencia será parecido)

In [2]:
import torch
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
import psutil

print("Cargando dataset Dolly 15k...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
SAMPLE_SIZE = 75
test_subset = dataset.shuffle(seed=42).select(range(SAMPLE_SIZE))

# --- PARÁMETROS ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
PROMPTS = [f"Instruction: {row['instruction']}\nResponse:" for row in test_subset]

# Configuración de generación idéntica para todos
GEN_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "top_k": 50,
    "do_sample": True
}

# Tabla para guardar resultados
results = []

# Determine if CUDA is available, otherwise use CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.float16
    print("CUDA está disponible. Usando GPU.")
else:
    print("CUDA no está disponible. Unsloth no puede usar CPU")

def get_vram_usage():
    """Devuelve la memoria RAM/VRAM usada en GB. Como en Unsolth no hay CPU se borra la otra parte"""
    if DEVICE == "cuda":
        return torch.cuda.max_memory_allocated() / (1024**3)

def clear_cache():
    """Limpia la memoria GPU entre pruebas, o no hace nada si es CPU"""
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats() # Only call if CUDA is available
    import gc
    gc.collect()

print(f"Modelo: {MODEL_ID}")
print(f"Número de pruebas (Prompts): {len(PROMPTS)}")
print(f"Ejemplo de prompt:\n{PROMPTS[0][:100]}...")

Cargando dataset Dolly 15k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

CUDA está disponible. Usando GPU.
Modelo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Número de pruebas (Prompts): 75
Ejemplo de prompt:
Instruction: Who were the children of the legendary Garth Greenhand, the High King of the First Men ...


# Unsloth

In [3]:
import os
import sys
from unsloth import FastLanguageModel
import time
import numpy as np
import torch

def benchmark_unsloth():
    print("\n--- Iniciando Benchmark: Unsloth ---")

    # Cargar modelo usando Unsloth
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = MODEL_ID,
        max_seq_length = 2048,
        dtype = None, # Python detecta automáticamente si usa float16/bfloat16
        load_in_4bit = False # Hace cuantización
    )

    FastLanguageModel.for_inference(model) # optimiza los grafos de cálculo

    # Warmup
    inputs = tokenizer("Warmup test", return_tensors="pt").to("cuda")
    model.generate(**inputs, max_new_tokens=5)
    clear_cache() # Reseteamos contadores tras la carga

    latencies = []
    tokens_per_sec = []

    # Loop
    for prompt in tqdm(PROMPTS, desc="Unsloth Inference"):
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        input_len = inputs.input_ids.shape[1]

        start_time = time.time()

        output = model.generate(**inputs, **GEN_CONFIG)
        end_time = time.time()

        # Calcular métricas
        gen_tokens = output[0].shape[0] - input_len
        duration = end_time - start_time

        latencies.append(duration)
        tokens_per_sec.append(gen_tokens / duration)

    peak_mem = get_vram_usage()

    # Guardar datos
    results.append({
        "Framework": "Unsloth",
        "Avg Latency (s)": np.mean(latencies),
        "Throughput (tok/s)": np.mean(tokens_per_sec),
        "Peak Memory (GB)": peak_mem
    })

    print(f"Unsloth completado.")

    # Limpieza final
    del model, tokenizer
    clear_cache()

benchmark_unsloth()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

--- Iniciando Benchmark: Unsloth ---
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth Inference: 100%|██████████| 75/75 [03:13<00:00,  2.58s/it]


Unsloth completado.


In [4]:
results

[{'Framework': 'Unsloth',
  'Avg Latency (s)': np.float64(2.5784404627482096),
  'Throughput (tok/s)': np.float64(36.867666682784865),
  'Peak Memory (GB)': 2.0778937339782715}]